# New Jupyter notebook to plot a chosen material's epsilon and mu values

First import necessary functions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Materials_Library_NEW import materials_data

Functions are imported since Jupyter notebooks sometimes have trouble accessing the functions from certain spots

In [2]:
def get_eps_mu_Michielssen(material_indices, frequencies):
    assert isinstance(material_indices, np.ndarray)
    assert isinstance(frequencies, np.ndarray)
    assert material_indices.ndim == 1
    assert frequencies.ndim == 1
    for material_index in material_indices:
        assert material_index in np.arange(1, 17)

    # Gets parameters from Michiellsen
    f = frequencies / 10**9  # in GHz
    M_epsr = np.vstack(
        [
            np.tile(
                np.array([10, 50, 15, 15, 15])[:, None], (1, len(f))
            ),  # Materials 1 to 5
            np.array(
                [  # Frequency-dependent permittivity for materials 6 to 8
                    5 / (f**0.861) - 1j * (8 / (f**0.569)),
                    8 / (f**0.778) - 1j * (10 / (f**0.682)),
                    10 / (f**0.778) - 1j * (6 / (f**0.861)),
                ]
            ),
            np.full((8, len(f)), 15, dtype=complex),  # Materials 9 to 16
        ]
    )

    # Fill constant values for permeability (mur)
    M_mur = np.vstack(
        [
            np.ones((2, len(f))),  # Materials 1 and 2
            np.array(
                [  # Frequency-dependent permeability for materials 3 to 5
                    5 / (f**0.974) - 1j * (10 / (f**0.961)),
                    3 / (f**1.0) - 1j * (15 / (f**0.957)),
                    7 / (f**1.0) - 1j * (12 / (f**1.0)),
                ]
            ),
            np.ones((3, len(f))),  # Materials 6 to 8
            np.array(
                [  # Frequency-dependent permeability for materials 9 to 16
                    (35 * (0.8**2)) / (f**2 + 0.8**2)
                    - 1j * (35 * 0.8 * f) / (f**2 + 0.8**2),
                    (35 * (0.5**2)) / (f**2 + 0.5**2)
                    - 1j * (35 * 0.5 * f) / (f**2 + 0.5**2),
                    (30 * (1**2)) / (f**2 + 1**2) - 1j * (30 * f) / (f**2 + 1**2),
                    (18 * (0.5**2)) / (f**2 + 0.5**2)
                    - 1j * (18 * 0.5 * f) / (f**2 + 0.5**2),
                    (20 * (1.5**2)) / (f**2 + 1.5**2)
                    - 1j * (20 * 1.5 * f) / (f**2 + 1.5**2),
                    (30 * (2.5**2)) / (f**2 + 2.5**2)
                    - 1j * (30 * 2.5 * f) / (f**2 + 2.5**2),
                    (30 * (2**2)) / (f**2 + 2**2) - 1j * (30 * 2 * f) / (f**2 + 2**2),
                    (25 * (3.5**2)) / (f**2 + 3.5**2)
                    - 1j * (25 * 3.5 * f) / (f**2 + 3.5**2),
                ]
            ),
        ]
    )

    # Initialize epsr and mur for the given material_indices
    eps_r = M_epsr[material_indices - 1, :]  # Python uses 0-based indexing
    mu_r = M_mur[material_indices - 1, :]

    return eps_r, mu_r


def get_eps_mus_real_materials(material_indices, frequencies):

    freq_min = min(frequencies)
    freq_max = max(frequencies)

    # 1. Initialize empty LISTS to store the arrays
    M_epsr_list = []
    M_mur_list = []

    for i in material_indices: # 'i' is the material index (e.g., 1, 2, 3...)
        
        # 2. Use 'i' to get the material's data dictionary
        # We subtract 1 to fix the "off-by-one" error
        try:
            material_data_entry = materials_data[i - 1]
        except IndexError:
            print(f"Material index {i} not found in materials_data (Index out of range).")
            continue
        except KeyError:
            print(f"Material index {i} not found in materials_data.")
            continue

        # 3. Use the data entry to check section and call the right function
        if material_data_entry['section']==1:
            # Pass the whole dictionary
            eps_r, mu_r = getEpAndMu_12_1(freq_min, freq_max, material_data_entry)
        elif material_data_entry['section']==4:
            eps_r, mu_r = getEpAndMu_12_4(freq_min, freq_max, material_data_entry)
        elif material_data_entry['section']==6:
            eps_r, mu_r = getEpAndMu_12_6(freq_min, freq_max, material_data_entry)
        elif material_data_entry['section']==7:
            eps_r, mu_r = getEpAndMu_12_7(freq_min, freq_max, material_data_entry)
        elif material_data_entry['section']==8:
            eps_r, mu_r = getEpAndMu_12_8(freq_min, freq_max, material_data_entry) 
        else:
            print(f'Material error: Unknown section for material index {i}')
            continue # Skip this material

        # 4. Append the resulting arrays to the lists
        M_epsr_list.append(eps_r)
    
    return eps_r, mu_r

# --- Helper functions for Section 4 ---
# MOVED HERE TO FIX THE NAMEERROR
def calculate_chi_m(f, params):
    # Use .get() for safety
    B = params.get('B', 0.0)
    C = params.get('C', 1.0) # Avoid divide by zero
    D = params.get('D', 1.0) # Avoid divide by zero
    
    j = 1j
    numerator = B * (1 - j * f / D)
    denominator = 1 - (f / C)**2 - (j * f / D)
    return np.divide(numerator, denominator, out=np.zeros_like(denominator, dtype=np.complex128), where=denominator!=0)

def calculate_epsilon1(f, params):
    """
    Calculates permittivity using the first model (ε1).
    """
    # Use .get() for safety
    B = params.get('B', 0.0)
    C = params.get('C', 0.0)
    D = params.get('D', 0.0)
    E = params.get('E', 0.0)
    F = params.get('F', 1.0) # Avoid divide by zero
    G = params.get('G', 1.0) # Avoid divide by zero
    
    j = 1j
    f_complex = f.astype(np.complex128)

    term1 = B
    term2 = C * np.power(f_complex, D)

    lorentz_num = E
    lorentz_den = 1 - (f / F)**2 + 2*j * (f / G)
    term3 = np.divide(lorentz_num, lorentz_den, out=np.zeros_like(lorentz_den, dtype=np.complex128), where=lorentz_den!=0)
    return term1 + term2 + term3

def calculate_epsilon2(f, params):
    """
    Calculates permittivity using the second model (ε2).
    """
    # Use .get() for safety
    B = params.get('B', 0.0)
    C = params.get('C', 0.0)
    D = params.get('D', 0.0)
    E = params.get('E', 0.0)
    F = params.get('F', 0.0)
    G = params.get('G', 1.0) # Avoid divide by zero
    H = params.get('H', 1.0) # Avoid divide by zero
    
    j = 1j
    f_complex = f.astype(np.complex128)

    term1 = B
    term2 = np.real(C) * np.power(f_complex, D)
    term3 = np.imag(C) * np.power(f_complex, E)

    lorentz_num = F
    lorentz_den = 1 - (f / G)**2 + 2*j * (f / H)
    term4 = np.divide(lorentz_num, lorentz_den, out=np.zeros_like(lorentz_den, dtype=np.complex128), where=lorentz_den!=0)
    return term1 + term2 + term3 + term4

# --- Main EpAndMu functions ---

def getEpAndMu_12_1(user_f_min, user_f_max, material):
    f_min, f_max = material['freq_range_ghz']
    if f_min is None or f_max is None:
        raise ValueError(f"Could not find frequency range for {material['name']}")

    # Use .get() for safety, providing 0.0 as a default
    params = material.get('eps_params', {})
    B = params.get('B', 0.0)
    C = params.get('C', 0.0)
    D = params.get('D', 0.0)
    G = params.get('G', 0.0)
    H = params.get('H', 0.0)
    I = params.get('I', 0.0)
    J = params.get('J', 0.0)
    
    num_points = 500
    frequencies = np.linspace(user_f_min, user_f_max, num_points)
    
    # Added a small value to avoid potential divide-by-zero in the formula
    epsilon_f = B + 2 * C * (frequencies ** D) + G * (1 - J * (frequencies - H)**2 + 1j * 2 * I * frequencies)**(-1)

    mu_f = np.ones(frequencies.shape)
    return(epsilon_f, mu_f)

def getEpAndMu_12_4(user_f_min, user_f_max, material):
    num_points = 500
    frequencies = np.linspace(user_f_min, user_f_max, num_points)
    
    if material.get('chi_m_params'):
        chi_m = calculate_chi_m(frequencies, material['chi_m_params'])
        mu_f = 1.0 + chi_m
    else:
        mu_f = np.ones(frequencies.shape, dtype=np.complex128)

    if material.get('eps1_params'):
        epsilon_f = calculate_epsilon1(frequencies, material['eps1_params'])
    elif material.get('eps2_params'):
        epsilon_f = calculate_epsilon2(frequencies, material['eps2_params'])
    else:
        epsilon_f = np.ones(frequencies.shape, dtype=np.complex128)

    return epsilon_f, mu_f

def getEpAndMu_12_6(user_f_min, user_f_max, material):
    f_min, f_max = material['freq_range_ghz']
    
    # Use .get() for safety
    params = material.get('eps_params', {})
    B = params.get('B', 0.0)
    C = params.get('C', 0.0)
    D = params.get('D', 0.0)
    E = params.get('E', 0.0)
    F = params.get('F', 0.0)
    G = params.get('G', 1.0) # Avoid divide by zero
    H = params.get('H', 1.0) # Avoid divide by zero
    
    num_points = 500
    frequencies = np.linspace(user_f_min, user_f_max, num_points)
    
    # Check for divide-by-zero potential
    denominator_term = (1 - (frequencies / G) ** 2 + 1j * 2 * frequencies / H)
    safe_denominator = np.where(denominator_term == 0, 1e-9, denominator_term) # Replace 0 with a small number
    
    epsilon_f = (B + np.real(C) * (frequencies ** D) + np.imag(C) * (frequencies ** E) + F / safe_denominator)

    mu_f = np.ones(frequencies.shape)
    return(epsilon_f, mu_f)

def getEpAndMu_12_7(user_f_min, user_f_max, material):
    f_min, f_max = material['freq_range_ghz']
    if f_min is None or f_max is None:
        raise ValueError(f"Could not find frequency range for {material['name']}")

    # Use .get() for safety
    params = material.get('eps_params', {})
    B = params.get('B', 0.0)
    C = params.get('C', 0.0)
    D = params.get('D', 0.0)
    G = params.get('G', 0.0)
    H = params.get('H', 0.0)
    I = params.get('I', 0.0)
    J = params.get('J', 0.0)
    
    num_points = 500
    frequencies = np.linspace(user_f_min, user_f_max, num_points)
    
    # Check for divide-by-zero potential
    denominator_term = (1 - J * (frequencies - H)**2 + 1j * 2 * I * frequencies)
    safe_denominator = np.where(denominator_term == 0, 1e-9, denominator_term)

    epsilon_f = B + 2 * C * (frequencies ** D) + G / safe_denominator

    mu_f = np.ones(frequencies.shape)
    return(epsilon_f, mu_f)

def getEpAndMu_12_8(user_f_min, user_f_max, material):
    f_min, f_max = material['freq_range_ghz']
    if f_min is None or f_max is None:
        raise ValueError(f"Could not find frequency range for {material['name']}")

    # Use .get() for safety
    params = material.get('eps_params', {})
    B = params.get('B', 0.0)
    C = params.get('C', 0.0)
    D = params.get('D', 0.0)
    G = params.get('G', 0.0)
    H = params.get('H', 0.0)
    I = params.get('I', 0.0)
    J = params.get('J', 0.0)
    
    num_points = 500
    frequencies = np.linspace(user_f_min, user_f_max, num_points)
    
    # Check for divide-by-zero potential
    denominator_term = (1 - (J *(frequencies - H)**2) + (1j*2*I*frequencies))
    safe_denominator = np.where(denominator_term == 0, 1e-9, denominator_term)

    epsilon_f = (B + (2*C*(frequencies**D)) + (G / safe_denominator))

    mu_f = np.ones(frequencies.shape)
    return(epsilon_f, mu_f)

Now the user is asked to identify what material they would like to plot and at what frequency range

In [ ]:
# Print options
i = 1
     
for material in materials_data:
    print(i, ". ", material['name'])
    i+=1
    
print('/n/n')

# Get input
mat_idx = [] 
mat_idx.append(int(input("Please select a material index from the list below: ")))
     
freq_min = float(input("Input min freq: "))
freq_max = float(input("Input max freq: "))

frequencies = np.linspace(freq_min, freq_max, 500)

# Get eps and mus via the same function called by the optimization

epsilon_f, mu_f = get_eps_mus_real_materials(mat_idx, frequencies)

# Plot the results

plt.figure()
plt.loglog(frequencies, np.real(epsilon_f), 'b-', linewidth=2, label='Re($\epsilon$)')
plt.loglog(frequencies, np.imag(epsilon_f), 'r--', linewidth=2, label='Im($\epsilon$)')
plt.xlabel('Frequency [GHz]', fontsize=12)
plt.ylabel('Epsilon', fontsize=12)
plt.legend(loc='best')
plt.grid(True, which="both", ls="--")
plt.title('Real and Imaginary Permittivity vs. Frequency', fontsize=14)
plt.xlim(1e-1, 1e3)
plt.ylim(1e-4, 10)
plt.show()

1 .  3D Quartz–alumina silica nitride 2.16 g/cc (8–40 GHz)
2 .  Alumina 99.5% dense (1–250 GHz)
3 .  Alumina 99.9% dense 3.86–3.90 g/cc (1–300 GHz)
4 .  Alumina 96–97% dense, 3.71 g/cc (5–250 GHz)
5 .  SRM709 (0.01–18 GHz), Lead oxide glass
6 .  Mullite 97% dense (2–35 GHz), 3 Al2O3 • 2SiO2
7 .  Magnesium oxide (MgO) (2–35 GHz)
8 .  Slip-cast silica, 2.05 g/cc (2–35 GHz)
9 .  Shuttle tile LI2200 (30–100 GHz)
10 .  Shuttle tile FRIC12 (3–100 GHz)
11 .  Beryllium oxide (BeO) (0.2–250 GHz)
12 .  Boron nitride; 2.28 g/cc (1–40 GHz) (Nominal εr = 4.08, Accumet Engineering)
13 .  Magnesium calcium titanate 30 (8–50 GHz) (Nominal εr = 30)
14 .  SRM 709 Lead-oxide glass (0.01–18 GHz)
15 .  SRM 710a Sodalime glass (0.01–18 GHz)
16 .  PyroCeram (2–40 GHz)
17 .  Sapphire wafer #1 (80–100 GHz)
18 .  Sapphire wafer #2 (80–100 GHz)
19 .  Silicon nitride 3.2–3.3 g/cc (2–35 GHz)
20 .  Fused silica-glass (Dynasil 4000) (2–40 GHz)
21 .  Fused silica glass (Dynasil 4000, 2.16–2.2 g/cc) (0.1–100 GHz)
22 .

<>:26: SyntaxWarning: invalid escape sequence '\e'
<>:27: SyntaxWarning: invalid escape sequence '\e'
<>:26: SyntaxWarning: invalid escape sequence '\e'
<>:27: SyntaxWarning: invalid escape sequence '\e'
C:\Users\21sme\AppData\Local\Temp\ipykernel_5028\3119944788.py:26: SyntaxWarning: invalid escape sequence '\e'
  plt.loglog(frequencies, np.real(epsilon_f), 'b-', linewidth=2, label='Re($\epsilon$)')
C:\Users\21sme\AppData\Local\Temp\ipykernel_5028\3119944788.py:27: SyntaxWarning: invalid escape sequence '\e'
  plt.loglog(frequencies, np.imag(epsilon_f), 'r--', linewidth=2, label='Im($\epsilon$)')
